# StormEngine V9.1 — pressure ablation and event completion

Run in order. The first stage completes frozen V9-A event metrics. The paired experiment then trains identical five-variable and six-variable models on 2015–2017, selects on 2018, and keeps 2019 locked until the validation gate passes.

In [ ]:
from pathlib import Path
import json, subprocess, sys
here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO
RUNNER = REPO / 'scripts' / 'run_v9_1_pressure_experiment.py'
DEVICE = 'cuda'
def run(mode, *extra):
    command = [sys.executable, '-u', str(RUNNER), mode, '--device', DEVICE, *extra]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)
print('Repository:', REPO)

## 1. Complete frozen V9-A 2025 event metrics

In [ ]:
run('events')

## 2. Paired data/model preflight and smoke

In [ ]:
run('preflight')
run('smoke')

## 3. Train both candidates with seeds 42 and 43

Runs are sequential on one GPU and resume from formal `last.pt` checkpoints after interruption.

In [ ]:
run('train')

## 4. Freeze and inspect 2018 validation

In [ ]:
run('validate')
validation_path = REPO / 'results' / 'v9_1_pressure_validation_2018' / 'benchmark.json'
validation = json.loads(validation_path.read_text(encoding='utf-8'))
print(json.dumps(validation['acceptance'], indent=2))

## 5. One-time 2019 test

Run only if the previous cell reports `passed: true`. This cell reads 2019 once and cannot be used for further tuning.

In [ ]:
ACKNOWLEDGE_ONE_TIME_2019 = False
if ACKNOWLEDGE_ONE_TIME_2019:
    run('test', '--acknowledge-one-time-2019')
else:
    print('2019 remains locked.')